# 04 - Đánh giá & So sánh mô hình
BTL môn Trí tuệ nhân tạo — Dữ liệu mô phỏng

## Bước 1: Load lại models và data test

In [ ]:
import joblib
import pandas as pd

X_train, X_test, y_train, y_test = joblib.load("../data/processed/train_test_data.pkl")
feature_names = joblib.load("../data/processed/feature_names.pkl")

model_names = ["Logistic_Regression", "Decision_Tree", "Random_Forest", "KNN", "SVM"]
models = {name: joblib.load(f"../results/models/{name}.pkl") for name in model_names}

## Bước 2: Bảng tổng hợp Accuracy / Precision / Recall / F1 (cả 2 lớp)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

detailed_results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    detailed_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 3),
        "Precision_Truot": round(precision_score(y_test, y_pred, pos_label=0), 3),
        "Recall_Truot": round(recall_score(y_test, y_pred, pos_label=0), 3),
        "F1_Truot": round(f1_score(y_test, y_pred, pos_label=0), 3),
        "Precision_Do": round(precision_score(y_test, y_pred, pos_label=1), 3),
        "Recall_Do": round(recall_score(y_test, y_pred, pos_label=1), 3),
        "F1_Do": round(f1_score(y_test, y_pred, pos_label=1), 3),
    })

detailed_df = pd.DataFrame(detailed_results)
print(detailed_df)
detailed_df.to_csv("../results/comparison_table.csv", index=False)

## Bước 3: classification_report chi tiết

In [ ]:
from sklearn.metrics import classification_report

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=["Trượt", "Đỗ"]))

## Bước 4: Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import os

os.makedirs("../results/figures", exist_ok=True)

for name, model in models.items():
    y_pred = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Trượt", "Đỗ"])
    plt.title(f"Confusion Matrix - {name}")
    plt.savefig(f"../results/figures/cm_{name}.png")
    plt.show()

## Bước 5: ROC Curve so sánh

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)
plt.title("So sánh ROC Curve giữa các mô hình")
plt.savefig("../results/figures/roc_comparison.png")
plt.show()

## Bước 6: Feature Importance (Random Forest)

In [ ]:
import seaborn as sns

rf_model = models["Random_Forest"]
importances = rf_model.feature_importances_

feat_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
feat_df = feat_df.sort_values(by="Importance", ascending=False)
print(feat_df)

plt.figure(figsize=(8, 6))
sns.barplot(data=feat_df, x="Importance", y="Feature")
plt.title("Mức độ ảnh hưởng của từng yếu tố đến Đỗ/Trượt")
plt.tight_layout()
plt.savefig("../results/figures/feature_importance.png")
plt.show()

## Bước 7: So sánh Accuracy và Recall_Truot giữa các mô hình

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=detailed_df, x="Model", y="Accuracy")
plt.title("So sánh Accuracy giữa các mô hình")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("../results/figures/accuracy_comparison.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=detailed_df, x="Model", y="Recall_Truot")
plt.title("So sánh Recall của lớp Trượt giữa các mô hình")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.axhline(y=0.5, color='red', linestyle='--', label='Ngưỡng tham khảo 0.5')
plt.legend()
plt.tight_layout()
plt.savefig("../results/figures/recall_truot_comparison.png")
plt.show()